# if_else

`if` and `else` are not beginner-only syntax. They are the core of system behavior selection: retry or fail, cache or fetch, allow or deny, alert or ignore. If your decision logic is unclear, your production behavior is unclear.


## 1. Problem First

Why does this matter in real systems?

- An API client must decide whether an error is retryable.
- A deployment script must branch on environment and safety checks.
- A log pipeline must decide which records to drop, repair, or escalate.


In [1]:
status_code = 503

if status_code >= 500:
    action = "retry"
else:
    action = "do_not_retry"

print(action)

retry


## 2. Minimal Concept

Core syntax:

- `if condition:` executes a block when the condition is truthy.
- `else:` executes the fallback block.
- Indentation defines which statements belong to each branch.


## 3. Mental Model

How Python actually behaves

Python evaluates the condition first, converts it through truthiness rules, and then executes exactly one branch. Branches are control-flow decisions, not just formatting blocks. The main engineering question is not "can this branch work?" but "can a reader predict system behavior under pressure?"


In [2]:
feature_flag = ""

if feature_flag:
    print("feature enabled")
else:
    print("feature disabled")

feature disabled


## 4. Internal Mechanics

An `if` statement compiles into conditional jumps in bytecode. Python is not doing magic here; it evaluates a value, checks truthiness, and jumps to the correct instruction block.


In [3]:
import dis

def classify_latency(latency_ms):
    if latency_ms > 500:
        return "slow"
    else:
        return "healthy"

dis.dis(classify_latency)
print(classify_latency(900))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (latency_ms)
              4 LOAD_CONST               1 (500)
              6 COMPARE_OP              68 (>)
             10 POP_JUMP_IF_FALSE        1 (to 14)

  5          12 RETURN_CONST             2 ('slow')

  7     >>   14 RETURN_CONST             3 ('healthy')
slow


## 5. Edge Cases

Where it breaks:

- Falsey values like `0`, `None`, `[]`, and `""` may mean different things but take the same branch.
- Mutating state inside one branch can make later logic harder to reason about.
- Deep nesting hides intent and makes it easy to miss missing cases.


In [4]:
timeout = 0

if timeout:
    print("custom timeout provided")
else:
    print("using default timeout")

configured_timeout = None
if configured_timeout is None:
    print("missing timeout config")

using default timeout
missing timeout config


## 6. Performance Thinking

- Condition checks are typically O(1).
- The expensive part is usually what each branch triggers: network calls, parsing, retries, writes.
- Clear early exits often reduce both runtime work and debugging cost.
- Poor branch design increases cognitive complexity even if CPU cost is trivial.


## 7. Real Use Case

A log ingestion service decides whether to alert immediately or just record the event based on severity.


In [5]:
record = {"level": "ERROR", "message": "database timeout"}

if record["level"] == "ERROR":
    destination = "pager"
else:
    destination = "storage"

print(destination)

pager


## 8. Anti-Pattern

What beginners do wrong:

- Depend on truthiness when the real question is `is None` or `== 0`.
- Put too much work inside branches without extracting intent into named functions.
- Write branches that are technically correct but impossible to audit quickly.


In [6]:
# Hard to reason about because 0 becomes "missing"
retries = 0
if retries:
    print("custom retries set")
else:
    print("using fallback retries")

# Better
if retries is None:
    print("missing retries config")
else:
    print("retries explicitly configured")

using fallback retries
retries explicitly configured


## 9. Interview Signals

Questions FAANG asks:

- What values are considered falsey in Python?
- When should you use `if value` versus `if value is None`?
- Why can overly clever branching be a production risk?
- How would you refactor nested conditionals into cleaner decision logic?


## 10. Exercise (Non-trivial)

You are building a CLI deploy command. It should only deploy if the target environment is valid, the build artifact exists, and the user has approval. Design the branching so each failure mode returns a clear reason instead of silently collapsing into one generic `else`.


In [7]:
def can_deploy(environment, artifact_exists, approved):
    if environment == "prod":
        if artifact_exists:
            if approved:
                return "deploy"
    return "block"

# Task:
# 1. Refactor this to avoid deep nesting.
# 2. Return specific failure reasons.
# 3. Explain which control-flow style is easier to maintain.